In [49]:
import numpy as np
import soundfile as sf
from scipy.signal import fftconvolve, resample_poly
import os
import math

In [50]:
print("CWD:", os.getcwd())
print("Files here:", os.listdir(".")[:20])

CWD: /Users/liann77/Desktop/NYUStudy/NYU26Spring/Colloquy/colluquyTeamWork/Surround-Stereo-Binaural-Custom-BRIR-based-Converter-/HRTF_Processor
Files here: ['.DS_Store', 'RIR.ipynb', 'Code.ipynb', 'compensation_filter.mat', 'Source', 'BRIR', '48K_24bit', 'RIR', 'extract_brir.ipynb']


In [51]:
audio, sr = sf.read("Source/TEST_AUDIO_44k.wav")

print("Original sample rate:", sr)

target_sr = 44100

if sr != target_sr:
    gcd = math.gcd(sr, target_sr)
    up = target_sr // gcd
    down = sr // gcd

    if audio.ndim == 1:
        audio_resampled = resample_poly(audio, up, down)
    else:
        audio_resampled = np.stack(
            [resample_poly(audio[:, i], up, down) for i in range(audio.shape[1])],
            axis=1
        )

    sf.write("Source/TEST_AUDIO_44k.wav", audio_resampled, target_sr)
    print("Downsampled to 44.1k and saved as TEST_AUDIO_44k.wav")
else:
    print("Audio already at 44.1k")

Original sample rate: 44100
Audio already at 44.1k


In [52]:
sweep_file = "Source/TEST_AUDIO_44k.wav"

sweep, fs = sf.read(sweep_file)

print("Sweep length:", len(sweep))
print("Sample rate:", fs)

Sweep length: 441002
Sample rate: 44100


In [53]:
inverse_sweep = sweep[::-1]
inverse_sweep = inverse_sweep / np.max(np.abs(inverse_sweep))

print("Inverse sweep created")

Inverse sweep created


In [ ]:
def extract_rir(record_file, inverse_sweep, fs, output_file):

    rec, fs_rec = sf.read(record_file)

    if fs_rec != fs:
        raise ValueError(f"Sample rate mismatch: {fs_rec} != {fs}")

    print("Processing:", record_file)

    if rec.ndim > 1:
        print("Input is multi-channel, converting to mono")
        rec = np.mean(rec, axis=1)

    # deconvolution
    rir = fftconvolve(rec, inverse_sweep, mode='full')

    # normalize
    rir = rir / np.max(np.abs(rir))

    sf.write(output_file, rir, fs)

    print("Saved:", output_file)

In [55]:
recordings = {
    "CenterRIR.wav": "RIR_C.wav",
    "LeftRIR.wav": "RIR_L.wav",
    "RightRIR.wav": "RIR_R.wav",
    "LeftRearRIR.wav": "RIR_LS.wav",
    "RightRearRIR.wav": "RIR_RS.wav",
    "SubRIR.wav": "RIR_SUB.wav",
}

recording_folder = "Source/RIRRecordings"
output_folder = "RIR"

os.makedirs(output_folder, exist_ok=True)

for rec, out in recordings.items():
    rec_path = os.path.join(recording_folder, rec)
    out_path = os.path.join(output_folder, out)

    extract_rir(
        rec_path,
        inverse_sweep,
        fs,
        out_path
    )

Processing: Source/RIRRecordings/CenterRIR.wav
Input is multi-channel, converting to mono
Saved: RIR/RIR_C.wav
Processing: Source/RIRRecordings/LeftRIR.wav
Input is multi-channel, converting to mono
Saved: RIR/RIR_L.wav
Processing: Source/RIRRecordings/RightRIR.wav
Input is multi-channel, converting to mono
Saved: RIR/RIR_R.wav
Processing: Source/RIRRecordings/LeftRearRIR.wav
Input is multi-channel, converting to mono
Saved: RIR/RIR_LS.wav
Processing: Source/RIRRecordings/RightRearRIR.wav
Input is multi-channel, converting to mono
Saved: RIR/RIR_RS.wav
Processing: Source/RIRRecordings/SubRIR.wav
Input is multi-channel, converting to mono
Saved: RIR/RIR_SUB.wav
